In [1]:
import os
import ast
import numpy as np
import pandas as pd
pd.set_option('display.max_rows', 500)
pd.set_option('display.width', 1000)

from tqdm import tqdm

from openai import AzureOpenAI

### Load OpenAI Model

In [2]:
os.environ["AZURE_OPENAI_KEY"] = ""
os.environ["AZURE_OPENAI_ENDPOINT"] = ""
os.environ["AZURE_API_VERSION"] = ""
os.environ["AZURE_DEPLOYMENT_ID"] = ""
os.environ["AWS_ACCESS_KEY"] = ""
os.environ["AWS_SECRET_KEY"] = ""
os.environ["AWS_SESSION_TOKEN"] = ""
model_name = "gpt-4o-mini"

client = AzureOpenAI(
    azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_KEY"),
    api_version=os.getenv("AZURE_API_VERSION"),
    azure_deployment=os.getenv("AZURE_DEPLOYMENT_ID")
)

### Taxonomy Functions

In [3]:
def clean_and_parse(x):
    import ast 

    if isinstance(x, str) and x.startswith('[') and x.endswith(']'):
        cleaned = x.replace('""', '"')
        cleaned = cleaned.replace('\n', ' ').replace('\r', '')  # Remove line breaks, if any

        try:
            return ast.literal_eval(cleaned)
        except Exception as e:
            print(f"Error parsing string: {cleaned}\n{e}")
            return x
    else:
        return x

def get_main_taxonomy_examples(dom: str, mode: str) -> str:

    if mode == "CC":
    
        df = pd.read_csv("../semeval-task-10/cc_taxonomy.csv")
        df = df.astype(str)
        df['Main Narrative Example'] = df['Main Narrative Example'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) and x.startswith('[') and x.endswith(']') else x)
                
        taxonomy = ''
        examples = ''

        dom = dom.split(":")[0] if len(dom.split(":")) > 0 else dom

        if dom in df['Main Narrative'].values:
            temp = df[df['Main Narrative'] == dom].reset_index()         
            taxonomy = taxonomy + f"Category: {temp.loc[0, 'Main Narrative']} | Definition: {temp.loc[0, 'Main Narrative Definition']}\n"
            if isinstance(temp.loc[0, 'Main Narrative Example'], list):
                for item in temp.loc[0, 'Main Narrative Example']:
                    examples = examples + f"{item} => {temp.loc[0, 'Main Narrative']}\n"
                if not temp.loc[0, 'Detail Instructions Main Narrative'] == 'nan':
                    examples = examples + f"Note: {temp.loc[0, 'Detail Instructions Main Narrative']}\n"
            else:
                pass


        return (taxonomy, examples)

    else:

        df = pd.read_csv("../semeval-task-10/urw_taxonomy.csv")
        df = df.astype(str)
        df['Main Narrative Example'] = df['Main Narrative Example'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) and x.startswith('[') and x.endswith(']') else x)
          
        taxonomy = ''
        examples = ''

        dom = dom.split(":")[0] if len(dom.split(":")) > 0 else dom

        if dom in df['Main Narrative'].values:
            temp = df[df['Main Narrative'] == dom].reset_index()           
            taxonomy = taxonomy + f"Category: {temp.loc[0, 'Main Narrative']} | Definition: {temp.loc[0, 'Main Narrative Definition']}\n"
            if isinstance(temp.loc[0, 'Main Narrative Example'], list):
                for item in temp.loc[0, 'Main Narrative Example']:
                    examples = examples + f"{item} => {temp.loc[0, 'Main Narrative']}\n"
                if not temp.loc[0, 'Detail Instructions Main Narrative'] == 'nan':
                    examples = examples + f"Note: {temp.loc[0, 'Detail Instructions Main Narrative']}\n"
            else:
                pass
            
        return (taxonomy, examples)

def get_sub_taxonomy_examples(sub: str, mode: str) -> str:

    if mode == "CC":
        df = pd.read_csv("../semeval-task-10/cc_taxonomy.csv")
        df = df.astype(str)
        df['Sub Narrative Example'] = df['Sub Narrative Example'].apply(clean_and_parse)

        taxonomy = ''
        examples = ''

        sub = sub.split(":")[2][1:] if len(sub.split(":")) > 1 else sub

        if sub in df['Sub Narrative'].values:
            temp = df[df['Sub Narrative'] == sub].reset_index()
            taxonomy = taxonomy + f"Category: {temp.loc[0, 'Sub Narrative']} | Definition: {temp.loc[0, 'Sub Narrative Definition']}\n"
            if isinstance(temp.loc[0, 'Sub Narrative Example'], list):
                for item in temp.loc[0, 'Sub Narrative Example']:
                    examples = examples + f"{item} => {temp.loc[0, 'Sub Narrative']}\n"
                if not temp.loc[0, 'Detail Instructions Sub Narrative'] == 'nan':
                    examples = examples + f"Note: {temp.loc[0, 'Detail Instructions Sub Narrative']}\n"
            else:
                pass
        return (taxonomy, examples)

    else:
        df = pd.read_csv("../semeval-task-10/urw_taxonomy.csv")
        df = df.astype(str)
        df['Sub Narrative Example'] = df['Sub Narrative Example'].apply(clean_and_parse)

        taxonomy = ''
        examples = ''

        sub = sub.split(":")[2][1:] if len(sub.split(":")) > 1 else sub

        if sub in df['Sub Narrative'].values:
            temp = df[df['Sub Narrative'] == sub].reset_index()
            taxonomy = taxonomy + f"Category: {temp.loc[0, 'Sub Narrative']} | Definition: {temp.loc[0, 'Sub Narrative Definition']}\n"
            if isinstance(temp.loc[0, 'Sub Narrative Example'], list):
                for item in temp.loc[0, 'Sub Narrative Example']:
                    examples = examples + f"{item} => {temp.loc[0, 'Sub Narrative']}\n"
                if not temp.loc[0, 'Detail Instructions Sub Narrative'] == 'nan':
                    examples = examples + f"Note: {temp.loc[0, 'Detail Instructions Sub Narrative']}\n"
            else:
                pass
        return (taxonomy, examples)

In [4]:
def create_prompt(text, dom, sub, mode):
    # Helper function replacing quotation marks in the text:
    replace_qm = lambda s: s.replace('"', "'")

    if mode == "CC":
        # Update predicted_labels by slicing from the 4th character
        main_taxonomy, main_examples = get_main_taxonomy_examples(dom, mode)
        sub_taxonomy, sub_examples = get_sub_taxonomy_examples(sub, mode)
        context = f"""You will be given an article along with the dominant narrative and sub narrative associated with the article. 
        
        GOAL: Justify the choice of dominant and sub narratives assigned to the article. Provide reasoning and quote relevant text from the original text as to why these are the correct choice of dominant and sub narratives for the text file.

            INSTRUCTIONS:
            Read the provided text carefully.
            You will be given: 
            1.Taxonomy - definition of the narrative
            2.List of relevant examples - sentences which determine which justify the narrative and align with its defintion 
            3.Any additional idenyifying information for the narratives.
            Based on the given information give an explanation as to why the dominant and sub narratives are the correct choice for the text file.
            
            Note: Keep the output concise and to the point - ideally find relevant textual examples.

            DOMINANT NARRATIVE: {dom[4:]}
            TAXONOMY: {main_taxonomy}
            RELEVANT EXAMPLES: 
            {main_examples}

            SUB NARRATIVE: {sub[4:]}
            TAXONOMY: {sub_taxonomy}
            RELEVANT EXAMPLES:
            {sub_examples}
        """

        prompt = f'''{context}
        -------------------------------------------------------
        Based on the given Instructions, Taxonomies and Examples: Justify the choice of dominant and sub narratives assigned to the Climate Change article. 

        ARTICLE TEXT TO PREDICT: "{replace_qm(text)}" => '''
        
        return {
            "role": "user",
            "content": prompt
        }
        
    else:
        # Update predicted_labels by slicing from the 5th character
        main_taxonomy, main_examples = get_main_taxonomy_examples(dom, mode)
        sub_taxonomy, sub_examples = get_sub_taxonomy_examples(sub, mode)

        context = f"""You will be given an article along with the dominant narrative and sub narrative associated with the article. 
        
        GOAL: Justify the choice of dominant and sub narratives assigned to the article. Provide reasoning and quote relevant text from the original text as to why these are the correct choice of dominant and sub narratives for the text file.

            INSTRUCTIONS:
            Read the provided text carefully.
            You will be given: 
            1.Taxonomy - definition of the narrative
            2.List of relevant examples - sentences which determine which justify the narrative and align with its defintion 
            3.Any additional idenyifying information for the narratives.
            Based on the given information give an explanation as to why the dominant and sub narratives are the correct choice for the text file.
            
            Note: Keep the output concise and to the point - ideally find relevant textual examples.

            DOMINANT NARRATIVE: {dom[5:]}
            TAXONOMY: {main_taxonomy}
            RELEVANT EXAMPLES: 
            {main_examples}

            SUB NARRATIVE: {sub[5:]}
            TAXONOMY: {sub_taxonomy}
            RELEVANT EXAMPLES:
            {sub_examples}
        """

        prompt = f'''{context}
        -------------------------------------------------------
        Based on the given Instructions, Taxonomies and Examples: Justify the choice of dominant and sub narratives assigned to the Ukraine Russia War article. 

        ARTICLE TEXT TO PREDICT: "{replace_qm(text)}" => '''
        
        return {
            "role": "user",
            "content": prompt
        }

In [5]:
def get_embedded_json(embedded_str):
    import re
    try:
        # Extract only the list content using regex
        match = re.search(r"\[.*\]", embedded_str)
        if not match:
            return []  # Return empty list if no valid list is found
        
        cleaned_str = match.group(0)  # Extract the matched list portion

        # Convert to Python list safely
        return ast.literal_eval(cleaned_str)
    
    except (ValueError, SyntaxError):
        return []  # Return empty list if parsing fails

In [15]:
def generate_response(text:str, dom, sub, mode, temp):
    system_main = \
'''You are an expert text generation model trained to analyse and justify the choice of dominant and sub narratives assigned to a given article.

Instructions:

Use ReACT (Reasoning and Contextual Text) to justify the choice of dominant and sub narratives assigned to the article.
Provide reasoning and quote relevant text from the original text as to why these are the correct choice of dominant and sub narratives for the text file.
Keep the output concise and to the point—ideally find relevant textual examples.
Provide justification in upto 80 words.

Categorization Rules:

Use the help of the provided taxonomy and examples to justify the choice of dominant and sub narratives assigned to the article in upto 80 words.

OUTPUT FORMAT:
Return text in paragraph(s) format.
'''

    
    prompt = create_prompt(text, dom, sub, mode)

    messages = [
        {"role": "system", "content": system_main},
        {"role": "user", "content": prompt.get("content", "")}
    ]

    response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    temperature=temp,
    max_tokens=100  
    )

    # output = get_embedded_json(str(response.choices[0].message.content))
    output = str(response.choices[0].message.content)
    return output


### Importing the data

In [16]:
df = pd.read_csv('SemEval 2025 Test Data/final_datasets/eng_data.csv')

In [17]:
df.shape

(68, 4)

In [18]:
df['mode'] = df['dominant_narrative'].apply(lambda x: 'CC' if x.split(':')[0] == 'CC' else 'URW')

In [19]:
df['mode'].value_counts()

mode
CC     34
URW    34
Name: count, dtype: int64

In [20]:
tqdm.pandas()

for i in np.arange(0.1, 1, 0.1):
    print("Iterating at temperature: ", i)
    temp = i.round(2)
    df[f'output_temp_{temp}'] = df.progress_apply(lambda x: generate_response(x['text'], x['dominant_narrative'], x['sub_narratives'], x['mode'], temp), axis=1)

Iterating at temperature:  0.1


100%|██████████| 68/68 [03:40<00:00,  3.24s/it]


Iterating at temperature:  0.2


100%|██████████| 68/68 [03:49<00:00,  3.38s/it]


Iterating at temperature:  0.30000000000000004


...

100%|██████████| 68/68 [03:36<00:00,  3.19s/it]


Iterating at temperature:  0.8


100%|██████████| 68/68 [03:37<00:00,  3.21s/it]


Iterating at temperature:  0.9


100%|██████████| 68/68 [03:36<00:00,  3.18s/it]


In [21]:
df.to_csv('SemEval 2025 Test Data/final_datasets/pred_eng_data.csv', index=False)

In [22]:
df.head()

,article_id,dominant_narrative,sub_narratives,text,mode,output_temp_0.1,output_temp_0.2,output_temp_0.30000000000000004,output_temp_0.4,output_temp_0.5,output_temp_0.6000000000000001,output_temp_0.7000000000000001,output_temp_0.8,output_temp_0.9
0,CC_TEST_00063.txt,CC: Hidden plots by secret schemes of powerful...,CC: Hidden plots by secret schemes of powerful...,“Sustainable Development Goals” are About Worl...,CC,The dominant narrative assigned to the article...,The dominant narrative assigned to the article...,The dominant narrative assigned to the article...,The dominant narrative assigned to the article...,"The dominant narrative of ""Hidden plots by sec...","The dominant narrative of ""Hidden plots by sec...","The dominant narrative of ""Hidden plots by sec...","The dominant narrative of ""Hidden plots by sec...",The dominant narrative assigned to the article...
1,RU_TEST_00021.txt,URW: Russia is the Victim,none,"Moscow To 'Mirror' West, NATO Approaches, Incl...",URW,The dominant narrative assigned to the article...,The dominant narrative assigned to the article...,"The dominant narrative of ""Russia is the Victi...","The dominant narrative of ""Russia is the Victi...","The dominant narrative of ""Russia is the Victi...",The dominant narrative assigned to the article...,"The dominant narrative of ""Russia is the Victi...","The dominant narrative of ""Russia is the Victi...","The choice of dominant narrative ""Russia is th..."
2,RU_TEST_00013.txt,URW: Blaming the war on others rather than the...,URW: Blaming the war on others rather than the...,My Take | Nato barbarians are expanding and ga...,URW,"The dominant narrative of ""Blaming the war on ...","The dominant narrative of ""Blaming the war on ...","The dominant narrative of ""Blaming the war on ...",The dominant narrative assigned to the article...,"The dominant narrative of ""Blaming the war on ...","The dominant narrative of ""Blaming the war on ...",The dominant narrative assigned to the article...,The dominant narrative assigned to the article...,"The dominant narrative of ""Blaming the war on ..."
3,EN_UA_DEV_25.txt,URW: Discrediting Ukraine,URW: Discrediting Ukraine: Ukraine is associat...,Understanding Ukrainian Nazism \n\nUnderstandi...,URW,The dominant narrative assigned to the article...,"The dominant narrative of ""Discrediting Ukrain...",The dominant narrative assigned to the article...,The dominant narrative assigned to the article...,"The dominant narrative of ""Discrediting Ukrain...","The dominant narrative of ""Discrediting Ukrain...",The dominant narrative assigned to the article...,"The choice of the dominant narrative ""Discredi...","The dominant narrative of ""Discrediting Ukrain..."
4,EN_UA_DEV_27.txt,URW: Discrediting Ukraine,none,NATO And Its Kiev Proxy in Last Roll of the Di...,URW,"The dominant narrative of ""Discrediting Ukrain...","The dominant narrative of ""Discrediting Ukrain...","The dominant narrative of ""Discrediting Ukrain...","The dominant narrative of ""Discrediting Ukrain...","The dominant narrative of ""Discrediting Ukrain...",The dominant narrative assigned to the article...,The dominant narrative assigned to the article...,"The dominant narrative of ""Discrediting Ukrain...",The article clearly aligns with the dominant n...


In [23]:
df.to_excel('SemEval 2025 Test Data/final_datasets/pred_eng_data.xlsx', index=False)